# UD2.01. Peticiones HTTP y lectura de JSON

**Módulo 5073 · Programación de Inteligencia Artificial · UD2 · Práctica P2.1**

Todos los servicios de inteligencia artificial de esta unidad se consumen igual: se construye
una petición HTTP, se envía y se lee un JSON. No hay más.

Antes de gastar cuota de un servicio de pago conviene dominar el mecanismo con APIs públicas que
no cuestan nada y no piden clave. Eso es este cuaderno.

Además vas a mirar el JSON **como lenguaje de marcado**: qué información lleva cada clave, en qué
se diferencia de una etiqueta o un atributo de XML, y por qué una respuesta anidada se recorre y
no se adivina. Es la continuación directa de la P1.2 de la UD1.

El enunciado completo, con los pesos de cada parte, está en la práctica
**P2.1 · Consumo de APIs REST y lectura de JSON**.

In [ ]:
!pip install -q requests

In [ ]:
import json
import time

import requests

print("requests", requests.__version__)

## 1. La primera petición, pieza a pieza

Una petición HTTP tiene cinco piezas: **método**, **URL**, **parámetros**, **cabeceras** y
**cuerpo**. Una petición `GET` sencilla solo usa las tres primeras.

Empezamos por la API abierta de datos del Gobierno de España, que no pide clave.

In [ ]:
URL = "https://datos.gob.es/apidata/catalog/dataset"

respuesta = requests.get(URL, params={"_pageSize": 5}, timeout=15)

print("Código de estado:", respuesta.status_code)
print("Content-Type:    ", respuesta.headers.get("Content-Type"))
print("Tamaño (bytes):  ", len(respuesta.content))
print("Ha tardado:      ", respuesta.elapsed.total_seconds(), "s")

Cuatro cosas antes de mirar el contenido, y las cuatro importan:

- **El código de estado** dice si puedes seguir. Si no es 2xx, el cuerpo no tiene lo que esperas.
- **El `Content-Type`** dice cómo interpretar el cuerpo. Si dijera `application/xml`, `.json()`
  fallaría y habría que usar un parser de XML como en la UD1.
- **El tamaño** avisa de respuestas gigantes antes de intentar imprimirlas.
- **El tiempo** es tu primera medida de latencia, y la vas a necesitar para decidir si una
  llamada cabe dentro de una interfaz interactiva.

`timeout=15` no es opcional. Sin él, una petición puede quedarse colgada indefinidamente.

### Los parámetros van en la URL

En una petición `GET` los parámetros viajan en la propia URL, después del `?`. `requests` los
construye por ti si le pasas `params=`, y además los codifica correctamente (los espacios, los
acentos, los caracteres reservados).

In [ ]:
respuesta = requests.get(URL, params={"_pageSize": 3, "_sort": "title"}, timeout=15)
print("URL final:", respuesta.url)

## 2. Los códigos de estado, provocados a propósito

La forma rápida de aprenderse los códigos es forzarlos. `httpbin.org` devuelve el código que le
pidas, y es el banco de pruebas estándar para esto.

In [ ]:
for codigo in (200, 400, 401, 404, 429, 500):
    r = requests.get(f"https://httpbin.org/status/{codigo}", timeout=15)
    print(f"{r.status_code}  ->  {'correcto' if r.ok else 'error'}")

La regla: **4xx es culpa tuya, 5xx es culpa del proveedor**. Con un 4xx no reintentes, arregla la
petición. La única excepción es el **429**, que significa "vas demasiado rápido" y sí se
reintenta después de esperar.

`raise_for_status()` convierte un código de error en una excepción, que es lo que quieres para
que un fallo no pase desapercibido.

In [ ]:
r = requests.get("https://httpbin.org/status/404", timeout=15)

try:
    r.raise_for_status()
except requests.HTTPError as error:
    print("Excepción capturada:", error)
    print("Código:", error.response.status_code)

### El tiempo agotado es otro tipo de fallo

Un `timeout` no devuelve ningún código: lanza una excepción distinta, porque no ha habido
respuesta. Hay que capturarlo aparte.

In [ ]:
try:
    requests.get("https://httpbin.org/delay/5", timeout=1)
except requests.Timeout:
    print("Tiempo agotado: el servicio no ha respondido a tiempo")
except requests.RequestException as error:
    print("Fallo de red:", type(error).__name__)

## 3. El JSON como lenguaje de marcado

En la UD1 viste JSON, XML, YAML y HTML como formas de **describir datos con marcas**. Aquí ese
mismo análisis se aplica a datos reales que llegan por la red.

Un documento JSON mezcla dos clases de claves, y distinguirlas es la mitad del trabajo:

- Las que **organizan** el documento: `results`, `items`, `data`, `_links`. No llevan
  información del dominio, llevan estructura.
- Las que **llevan el dato**: `title`, `amount`, `date`, `confidenceScore`.

En XML esa diferencia se ve porque unas cosas son etiquetas y otras atributos. JSON no distingue:
todo son pares clave-valor. Gana en simplicidad y pierde en expresividad, y por eso XML sigue
vivo en la Administración, donde importa poder decir que un valor tiene un tipo y un espacio de
nombres.

In [ ]:
r = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu", timeout=15)
r.raise_for_status()
datos = r.json()

print("Tipo del resultado:", type(datos))
print("Claves de primer nivel:", len(datos))
print(sorted(datos)[:12])

### Una herramienta que vas a usar toda la unidad

Antes de escribir la extracción de un dato, hay que ver la forma del documento. Esta función
recorre cualquier JSON e imprime el camino de cada hoja con su tipo.

Cuando llegues a las respuestas de Azure, que anidan cuatro y cinco niveles, esta función es la
diferencia entre leer la documentación y adivinar.

In [ ]:
def describe(valor, prefijo="", profundidad=3):
    """Imprime el camino y el tipo de cada hoja de un JSON.

    profundidad limita cuánto se baja, para que una respuesta grande no
    llene la pantalla.
    """
    if profundidad == 0:
        print(f"{prefijo} -> {type(valor).__name__} (cortado)")
        return

    if isinstance(valor, dict):
        for clave, sub in valor.items():
            describe(sub, f"{prefijo}.{clave}" if prefijo else clave, profundidad - 1)
    elif isinstance(valor, list):
        if not valor:
            print(f"{prefijo}[] -> lista vacía")
        else:
            describe(valor[0], f"{prefijo}[0]", profundidad - 1)
            if len(valor) > 1:
                print(f"{prefijo}[...] -> {len(valor)} elementos en total")
    else:
        print(f"{prefijo} -> {type(valor).__name__} = {repr(valor)[:60]}")

In [ ]:
describe({k: datos[k] for k in ("name", "height", "weight", "types")})

Fíjate en lo que dice esa salida: para llegar al nombre de un tipo hay que recorrer
`types[0].type.name`. Tres niveles para un dato que en la interfaz es una palabra.

Esto no es un capricho de la API: es lo normal. Las respuestas anidadas existen porque cada nivel
añade información que a ti no te interesa hoy pero a otro consumidor sí.

### Ausente no es lo mismo que `null`

Dos situaciones distintas que se confunden siempre:

- La clave **no está** en el documento → en Python, un `KeyError`.
- La clave está y su valor es **`null`** → en Python, `None`.

La primera suele significar "esta API no da ese dato en este caso". La segunda, "el dato existe
como concepto pero está vacío". Y se tratan distinto.

In [ ]:
ejemplo = {"nombre": "Pikachu", "apodo": None}

print("apodo está en el documento:", "apodo" in ejemplo)
print("apodo vale:", ejemplo["apodo"])
print("mote está en el documento:", "mote" in ejemplo)

# .get() no distingue los dos casos: los dos devuelven None
print(ejemplo.get("apodo"), ejemplo.get("mote"))

# Si necesitas distinguirlos, hay que preguntar por la clave
print(ejemplo.get("mote", "AUSENTE"), ejemplo.get("apodo", "AUSENTE"))

### Las fechas viajan como cadena

JSON no tiene tipo fecha. Las fechas viajan como texto, casi siempre en formato ISO 8601, y hay
que convertirlas antes de poder comparar u ordenar.

In [ ]:
from datetime import datetime

fecha_texto = "2026-09-08T10:30:00Z"

# fromisoformat no acepta la Z de UTC hasta Python 3.11; esto funciona siempre
fecha = datetime.fromisoformat(fecha_texto.replace("Z", "+00:00"))

print(fecha, type(fecha))
print("Año:", fecha.year, "| Mes:", fecha.month)

## 4. Extracción robusta de respuestas anidadas

Escribir `datos["types"][0]["type"]["name"]` funciona hasta el día en que un elemento no trae
esa clave, y entonces la aplicación entera se cae por un dato que ni siquiera era importante.

Hay tres formas de protegerse. Cada una tiene su momento.

In [ ]:
def extrae_con_try(pokemon):
    """Directo y legible. Bien cuando el dato es obligatorio."""
    try:
        return pokemon["types"][0]["type"]["name"]
    except (KeyError, IndexError, TypeError):
        return None


def extrae_con_get(pokemon):
    """Encadenar .get() se vuelve ilegible en cuanto hay tres niveles."""
    tipos = pokemon.get("types") or []
    if not tipos:
        return None
    return (tipos[0].get("type") or {}).get("name")


def camino(estructura, ruta, defecto=None):
    """Recorre un camino tipo 'types.0.type.name' sin romperse por el camino."""
    actual = estructura
    for paso in ruta.split("."):
        try:
            actual = actual[int(paso)] if paso.isdigit() else actual[paso]
        except (KeyError, IndexError, TypeError):
            return defecto
    return actual


print(extrae_con_try(datos))
print(extrae_con_get(datos))
print(camino(datos, "types.0.type.name"))
print(camino(datos, "types.99.type.name", defecto="no hay"))

La tercera es la que compensa cuando extraes muchos campos de respuestas profundas, que es
exactamente el caso de los servicios cognitivos. Las otras dos son mejores cuando son uno o dos
campos y quieres que se vea qué estás cogiendo.

### Un lote, y que un fallo no se lleve el resto

Cuando pides varios elementos, uno va a fallar. La estructura correcta es separar lo que ha ido
bien de lo que no, con el motivo, y seguir.

In [ ]:
NOMBRES = ["pikachu", "charmander", "no-existe-este", "snorlax"]

obtenidos, rechazados = [], []

for nombre in NOMBRES:
    try:
        r = requests.get(f"https://pokeapi.co/api/v2/pokemon/{nombre}", timeout=15)
        r.raise_for_status()
        p = r.json()
        obtenidos.append({
            "nombre": p["name"],
            "altura_cm": p["height"] * 10,
            "peso_kg": p["weight"] / 10,
            "tipos": [t["type"]["name"] for t in p.get("types", [])],
        })
    except requests.HTTPError as error:
        rechazados.append((nombre, f"HTTP {error.response.status_code}"))
    except requests.RequestException as error:
        rechazados.append((nombre, f"red: {type(error).__name__}"))

print("Obtenidos:", len(obtenidos))
for o in obtenidos:
    print("  ", o)
print("Rechazados:", rechazados)

Fíjate en que las claves del diccionario de salida **las has elegido tú**: `altura_cm` en lugar
de `height`, con la unidad en el nombre y el valor ya convertido. Eso es diseñar un formato, y
es una decisión tan técnica como el resto.

## 5. POST: cabeceras y cuerpo

Las llamadas a servicios de IA son casi siempre `POST` con cuerpo JSON. `httpbin.org/post`
devuelve exactamente lo que le mandas, así que sirve para ver qué se envía de verdad.

In [ ]:
cuerpo = {"kind": "LanguageDetection", "texto": "Bon dia, això és valencià"}

r = requests.post("https://httpbin.org/post", json=cuerpo, timeout=15)
eco = r.json()

print("Content-Type enviado:", eco["headers"]["Content-Type"])
print("Cuerpo recibido:     ", eco["json"])

Ahora lo mismo con `data=`, que es lo que se hace cuando no se conoce el atajo:

In [ ]:
r = requests.post("https://httpbin.org/post", data=json.dumps(cuerpo), timeout=15)
eco = r.json()

print("Content-Type enviado:", eco["headers"].get("Content-Type", "(ninguno)"))
print("Cuerpo en crudo:     ", eco["data"][:60])

Ahí está la diferencia, y es la que importa: con `data=` **no se ha enviado ninguna cabecera
`Content-Type`**. El cuerpo va, pero sin decir de qué tipo es.

`httpbin` es permisivo y lo interpreta igual. Un servicio real no: responde **415, tipo de
contenido no soportado**, y es el error que más tiempo hace perder al empezar.

**`json=` hace las dos cosas**: serializa el diccionario y pone la cabecera correcta. Con `data=`
hay que acordarse de la segunda.

`data=` sí es la forma correcta cuando el cuerpo **no es JSON**: los bytes de una imagen o de un
audio, que es lo que harás en los cuadernos 04 y 05.

### Cabeceras propias

Las claves de API viajan en una cabecera. Aquí solo se comprueba que llegan; en el cuaderno
siguiente se verá de dónde sale el valor, que nunca es del código.

In [ ]:
r = requests.post(
    "https://httpbin.org/post",
    json={"texto": "prueba"},
    headers={"Ocp-Apim-Subscription-Key": "valor-de-prueba", "X-Origen": "UD2"},
    timeout=15,
)
recibidas = r.json()["headers"]

print("X-Origen:", recibidas.get("X-Origen"))
print("La cabecera de clave ha llegado:", "Ocp-Apim-Subscription-Key" in recibidas)

## 6. La función que reutilizarás el resto de la unidad

Todo lo anterior se junta aquí. A partir de este punto, ninguna llamada de la unidad se escribe
a pelo: se hace a través de esta función, que se ampliará en la P2.2 con reintentos.

Lo que aporta: distingue las cuatro clases de fallo y las convierte en excepciones propias con
un mensaje legible. Quien llama decide qué enseñar al usuario; la función no imprime nada.

In [ ]:
class ErrorServicio(Exception):
    """Cualquier fallo al hablar con un servicio remoto."""


class ErrorAutenticacion(ErrorServicio):
    """401 o 403: la clave falta, es incorrecta o no cubre ese servicio."""


class ErrorCuota(ErrorServicio):
    """429: demasiadas peticiones. Es el único 4xx que se reintenta."""


class ErrorPeticion(ErrorServicio):
    """Resto de 4xx: la petición está mal construida. No reintentar."""


class ErrorProveedor(ErrorServicio):
    """5xx o fallo de red: el problema no está en tu código."""


def peticion_json(url, *, metodo="GET", cabeceras=None, cuerpo=None,
                  parametros=None, timeout=15):
    """Hace la petición, comprueba el estado y devuelve el JSON ya convertido."""
    try:
        respuesta = requests.request(
            metodo, url, headers=cabeceras, json=cuerpo,
            params=parametros, timeout=timeout,
        )
    except requests.Timeout as error:
        raise ErrorProveedor(f"El servicio no respondió en {timeout} s") from error
    except requests.RequestException as error:
        raise ErrorProveedor(f"No se pudo contactar con el servicio: {error}") from error

    codigo = respuesta.status_code
    if codigo in (401, 403):
        raise ErrorAutenticacion(f"Credenciales rechazadas (HTTP {codigo})")
    if codigo == 429:
        espera = respuesta.headers.get("Retry-After", "desconocido")
        raise ErrorCuota(f"Límite de peticiones superado. Retry-After: {espera}")
    if 400 <= codigo < 500:
        raise ErrorPeticion(f"Petición incorrecta (HTTP {codigo}): {respuesta.text[:200]}")
    if codigo >= 500:
        raise ErrorProveedor(f"Fallo del proveedor (HTTP {codigo})")

    try:
        return respuesta.json()
    except ValueError as error:
        raise ErrorServicio(
            f"La respuesta no es JSON. Content-Type: "
            f"{respuesta.headers.get('Content-Type')}"
        ) from error

In [ ]:
# Funciona
print(peticion_json("https://pokeapi.co/api/v2/pokemon/ditto")["name"])

# Y cada fallo se distingue
for url, etiqueta in [
    ("https://httpbin.org/status/401", "clave mala"),
    ("https://httpbin.org/status/429", "demasiadas peticiones"),
    ("https://httpbin.org/status/400", "petición mal formada"),
    ("https://httpbin.org/status/503", "servicio caído"),
    ("https://httpbin.org/html", "respuesta que no es JSON"),
]:
    try:
        peticion_json(url)
    except ErrorServicio as error:
        print(f"{etiqueta:26} -> {type(error).__name__}: {error}")

Compara esa salida con lo que vería un usuario si dejaras salir la traza de `requests`. Esa es
toda la diferencia entre un ejercicio y algo utilizable, y es lo que se corrige en la práctica.

## Para la entrega

Este cuaderno es el material guiado. Tu entrega es **tu propio cuaderno** con las cinco partes
de la práctica P2.1 resueltas, que reutilizan todo lo de aquí:

1. La petición pieza a pieza, con los tres errores provocados y cuál reintentarías.
2. El análisis del JSON como lenguaje de marcado, con la tabla de claves y la columna
   "estructura o contenido".
3. La extracción robusta sobre cinco elementos, uno de ellos inexistente.
4. `POST`, cabeceras y tu versión de `peticion_json()`.
5. La primera llamada real a Azure AI Language, con el `.env` ya montado.

Para la parte 5 necesitas el cuaderno siguiente, **UD2.02**, que es el de la gestión de
secretos. Ese orden no es casual: la clave se guarda bien **antes** de la primera llamada real,
no después.